# B0 — Conjunto de TEST estratificado por etiqueta

Genera un conjunto de **test independiente** de los 100 registros de desarrollo,
estratificado por etiqueta N2 (no por boletín), para una evaluación de
generalización honesta del clasificador B0.

**Por qué estratificar por etiqueta**: el dominio B0 es ~3,4 % del corpus; un
muestreo aleatorio sería casi todo negativo e inútil para medir las 8 etiquetas
N2. Igual que en el muestreo de desarrollo, se sobre-representan deliberadamente
las publicaciones relevantes y cada etiqueta.

**Independencia**: se excluyen por *descripción* todos los registros del GT100 (su
`id` está reindexado 0–99 y no coincide con el del corpus) y se deduplica el
corpus por texto. Semilla propia distinta del dev (dev=42) para extraer filas
frescas.

> Las señales léxicas solo SELECCIONAN candidatos; la anotación
> (`is_relevant_gt`, `procedures_gt`, `technologies_gt`) se hace a mano después.

In [1]:
import html
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from clasificador.agent import inferir_act_type

SEED_TEST = 13062026  # distinta del muestreo de desarrollo (42)
PATH_CORPUS = ROOT / "data/raw/silver_official_gazettes_2025_Q1.parquet"
PATH_GT100 = ROOT / "data/ground_truth/ground_truth_100_anotado.csv"
PATH_OUT = ROOT / "data/ground_truth/test_b0_para_anotar.csv"

In [2]:
df = pd.read_parquet(PATH_CORPUS)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else "")
df["bulletin"] = df["bulletin"].str.lower()
print(f"Corpus: {len(df):,} filas")

gt = pd.read_csv(PATH_GT100)
gt_desc = set(gt["description"].apply(lambda x: html.unescape(str(x))))
print(f"GT100: {len(gt)} filas, {len(gt_desc)} descripciones únicas")

Corpus: 65,201 filas
GT100: 100 filas, 100 descripciones únicas


In [3]:
# Excluir GT100 POR DESCRIPCIÓN (el id del GT100 está reindexado) + dedup por texto
n0 = len(df)
df = df[~df["description"].isin(gt_desc)].copy()
df = df.drop_duplicates(subset="description", keep="first").reset_index(drop=True)
print(f"Tras excluir GT100 y deduplicar: {len(df):,} filas (descartadas {n0-len(df):,})")

Tras excluir GT100 y deduplicar: 56,390 filas (descartadas 8,811)


In [4]:
# Señales léxicas por etiqueta (solo para SELECCIONAR candidatos a anotar)
d = df["description"].str.lower()
mascaras = {
    "AAP_AAC": d.str.contains("previa y de construcción", na=False, regex=False),
    "AAU": df["contains_aau"].fillna(False).astype(bool)
           | d.str.contains("autorización ambiental unificada", na=False, regex=False),
    "AAI": d.str.contains("autorización ambiental integrada", na=False, regex=False)
           | d.str.contains("ippc", na=False, regex=False),
    "IAE": d.str.contains("ambiental estratégic", na=False, regex=False),
    "DUP": d.str.contains("utilidad pública", na=False, regex=False),
    "DIA": d.str.contains("declaración de impacto ambiental", na=False, regex=False),
    "IIA": d.str.contains("informe de impacto ambiental", na=False, regex=False),
    "AAP": d.str.contains("autorización administrativa previa", na=False, regex=False),
    "AAC": d.str.contains("autorización administrativa de construcción", na=False, regex=False)
           | d.str.contains("autorización de construcción", na=False, regex=False),
}
# Cuotas (~38 total, ~21% negativos), espejando proporciones del dev
CUOTAS = {"AAP_AAC":3, "AAU":3, "AAI":3, "IAE":3, "DUP":3, "DIA":4, "IIA":3, "AAP":4, "AAC":4}
# Orden en cascada: los casos conjuntos (AAP_AAC) y etiquetas raras primero
ORDEN = ["AAP_AAC", "AAU", "AAI", "IAE", "DUP", "DIA", "IIA", "AAP", "AAC"]
N_NEGATIVOS = 8

In [5]:
sampled_idx = set()
frames = []
for lab in ORDEN:
    pool = df[mascaras[lab] & ~df.index.isin(sampled_idx)]
    n = min(CUOTAS[lab], len(pool))
    s = pool.sample(n, random_state=SEED_TEST).copy()
    s["candidato_a"] = lab
    sampled_idx.update(s.index.tolist())
    frames.append(s)
    print(f"  {lab:<8} pool={len(pool):>5}  sampled={n}")

# Negativos: ninguna señal de dominio B0
any_kw = pd.concat(list(mascaras.values()), axis=1).any(axis=1)
pool_neg = df[~any_kw & ~df.index.isin(sampled_idx)]
neg = pool_neg.sample(N_NEGATIVOS, random_state=SEED_TEST).copy()
neg["candidato_a"] = "NEGATIVO"
frames.append(neg)
print(f"  {'NEGATIVO':<8} pool={len(pool_neg):>5}  sampled={N_NEGATIVOS}")

test = pd.concat(frames, ignore_index=True)
test["act_type_n1"] = [inferir_act_type(r.description, r.bulletin).value for r in test.itertuples()]
for c in ("is_relevant_gt", "procedures_gt", "technologies_gt"):
    test[c] = ""
test = test[["id","bulletin","description","candidato_a","act_type_n1",
             "is_relevant_gt","procedures_gt","technologies_gt"]]
print(f"\nTotal test: {len(test)}")

  AAP_AAC  pool=  239  sampled=3
  AAU      pool=  211  sampled=3
  AAI      pool=  260  sampled=3
  IAE      pool=  311  sampled=3
  DUP      pool=  662  sampled=3
  DIA      pool=  274  sampled=4
  IIA      pool=  512  sampled=3
  AAP      pool=  981  sampled=4
  AAC      pool=  521  sampled=4
  NEGATIVO pool=53514  sampled=8

Total test: 38


In [6]:
# Verificación: reparto, multi-boletín, independencia, sin duplicados
print("Reparto por candidato_a:")
print(test["candidato_a"].value_counts().to_string())
print(f"\nBoletines distintos: {test['bulletin'].nunique()}")
print(test["bulletin"].value_counts().to_string())
print(f"\nSolapamiento por descripción con GT100: {test['description'].isin(gt_desc).sum()} (debe ser 0)")
print(f"Duplicados internos de descripción: {test['description'].duplicated().sum()} (debe ser 0)")
assert test["description"].isin(gt_desc).sum() == 0
assert test["description"].duplicated().sum() == 0
assert test["bulletin"].nunique() > 1, "el test debe ser multi-boletín"
print("\nVerificación OK")

Reparto por candidato_a:
candidato_a
NEGATIVO    8
DIA         4
AAP         4
AAC         4
AAP_AAC     3
AAU         3
AAI         3
IAE         3
DUP         3
IIA         3

Boletines distintos: 16
bulletin
boe                5
bocyl              5
dog                4
docm               4
bon                3
boja               3
bocm               2
bopv               2
bopa               2
doe                2
boc                1
bor                1
boib               1
boa                1
madridambiental    1
dogc               1

Solapamiento por descripción con GT100: 0 (debe ser 0)
Duplicados internos de descripción: 0 (debe ser 0)

Verificación OK


In [7]:
# Revisión cualitativa: 2 ejemplos por candidato
for lab in test["candidato_a"].unique():
    print(f"--- {lab} ---")
    for _, r in test[test["candidato_a"] == lab].head(2).iterrows():
        print(f"  [{r['bulletin'].upper()}] {str(r['description'])[:110]}")
    print()

--- AAP_AAC ---
  [DOG] RESOLUCIÓN de 30 de diciembre de 2024, del Departamento Territorial de Ourense, por la que se conceden las aut
  [DOG] RESOLUCIÓN de 7 de marzo  de 2025, del Departamento Territorial de Ourense, por la que se concede la autorizac

--- AAU ---
  [BON] INFORMACIÓN PÚBLICA. Expediente de modificación sustancial de autorización ambiental unificada promovido por M
  [BOJA] Resolución de 6 de marzo de 2025, de la Delegación Territorial de Sostenibilidad, Medio Ambiente y Economía Az

--- AAI ---
  [BOCM] Impacto ambiental
– Resolución de 13 de diciembre de 2024, de la Directora General de Transición Energética y 
  [BOC] Anuncio de dictado de Resolución por la que se otorga autorización para una Modificación No Sustancial de la A

--- IAE ---
  [BOPV] RESOLUCIÓN de 28 de enero de 2025, del Director de Administración Ambiental, por la que se formula informe amb
  [BOJA] Resolución de 13 de febrero de 2025, de la Delegación Territorial de Sostenibilidad, Medio Ambiente 

In [8]:
PATH_OUT.parent.mkdir(parents=True, exist_ok=True)
test.to_csv(PATH_OUT, index=False)
print(f"Guardado: {PATH_OUT}  ({len(test)} registros)")
print("Siguiente: anotar a mano is_relevant_gt, procedures_gt y technologies_gt,")
print("y devolverlo como data/ground_truth/test_b0_anotado.csv (la columna")
print("candidato_a se puede borrar tras anotar).")

Guardado: /home/hugo/Documents/Obsidian Vault/Codigo/data/ground_truth/test_b0_para_anotar.csv  (38 registros)
Siguiente: anotar a mano is_relevant_gt, procedures_gt y technologies_gt,
y devolverlo como data/ground_truth/test_b0_anotado.csv (la columna
candidato_a se puede borrar tras anotar).
